# CIFAR-10 Demo: Image Classification with CREST

This notebook walks through a complete CREST workflow on the CIFAR-10 image
classification task. It covers the typical lifecycle of a CREST model:

1. **Loading data** — CIFAR-10 via `tf.keras.datasets`, wrapped in `StructuredDataset`
2. **Building a CNN** — two `Node`s composed in a `HierarchalTensorGraph`
3. **Compiling and training** with `Model.compile` / `Model.fit`
4. **Evaluating** with `Model.evaluate` and CREST `Metrics`
5. **Predictions** — via `Batcher`, raw dict input, and with extra output coordinates
6. **Saving and reloading** a trained model with `Model.save` / `Model.load`

CIFAR-10 consists of 60 000 32×32 RGB images across 10 object classes
(airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).
Unlike MNIST, images already carry three colour channels, so the preprocessing
node passes them through unchanged rather than adding a channel dimension.

In [ ]:
# Allow crest to be imported from the examples/ sub-directory
%cd -q ..

## 1. Loading CIFAR-10 Data

`tf.keras.datasets.cifar10.load_data()` downloads the dataset on first call and
returns two tuples: `(x_train, y_train)` with 50 000 samples and `(x_test, y_test)`
with 10 000 samples. Raw images are `uint8` arrays of shape `(N, 32, 32, 3)`;
labels are integer arrays of shape `(N, 1)` that we flatten to `(N,)`.

We split the 50 000 training samples into **45 000 train** and **5 000 validation**,
keep all 10 000 test samples, and derive a label-free *prediction* set from the
test images (no ground-truth labels). Each split is wrapped in a `StructuredDataset`
using the same label convention as the MNIST demo: `labels = ['image', 'class']`.

In [ ]:
import numpy as np
import tensorflow as tf

from crest.data.loading import StructuredDataset

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck',
]

(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Flatten labels from (N, 1) to (N,)
y_train_full = y_train_full.flatten()
y_test       = y_test.flatten()

# Train / validation split
n_valid  = 5_000
x_train, y_train = x_train_full[:-n_valid], y_train_full[:-n_valid]
x_valid, y_valid = x_train_full[-n_valid:], y_train_full[-n_valid:]

print(f"Train : {x_train.shape}  labels {y_train.shape}")
print(f"Valid : {x_valid.shape}  labels {y_valid.shape}")
print(f"Test  : {x_test.shape}   labels {y_test.shape}")

In [ ]:
INP = 'image'
OUT = 'class'
n_classes  = 10
patch_size = 32
n_channels = 3

print(f"Labels     : ['{INP}', '{OUT}']")
print(f"Image shape: ({patch_size}, {patch_size}, {n_channels})")
print(f"Classes    : {CIFAR10_CLASSES}")

In [ ]:
# Raw images are uint8 in [0, 255]; normalize to float32 in [0, 1] because
# TensorSpec defaults to dtype='float32'
def _norm(arr):
    return arr.astype('float32') / 255.0

ds_train = StructuredDataset(_norm(x_train), y_train, labels=[INP, OUT])
ds_valid = StructuredDataset(_norm(x_valid), y_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(_norm(x_test),  y_test,  labels=[INP, OUT])
ds_pred  = StructuredDataset(_norm(x_test),           labels=[INP])

print(f"After normalization:")
print(f"Train : images {ds_train.data[0].shape}  dtype {ds_train.data[0].dtype}  "
      f"range [{ds_train.data[0].min():.2f}, {ds_train.data[0].max():.2f}]")
print(f"Valid : images {ds_valid.data[0].shape}  labels {ds_valid.data[1].shape}")
print(f"Test  : images {ds_test.data[0].shape}   labels {ds_test.data[1].shape}")
print(f"Pred  : images {ds_pred.data[0].shape}")

## 2. Building an Improved CNN as a CREST HTG

A `HierarchalTensorGraph` (HTG) composes one or more `Node`s into a coupled system.
Each `Node` wraps a Python callable and declares its tensor interface via `TensorSpec` .

We use two nodes chained through an intermediate key `'x'`:

- **preprocessing** — confirms the channel layout and applies random horizontal flip
- **cnn** — two convolutional blocks with batch normalization, followed by a dense head

Note: Three architectural improvements over a minimal baseline push accuracy from ~45 % toward ~75 %:

### 2.1 Batch Normalization

`BatchNormalization` is inserted **between each Conv2D and its ReLU activation**
(`Conv → BN → ReLU`). It normalizes each feature map to zero mean and unit variance
across the mini-batch, then applies learned scale and shift parameters.


### 2.2 Second Convolutional Block

A second block of `Conv2D(128)` + `MaxPooling2D` is added after the first block
(which uses 32 and 64 filters).


### 2.3 Random Horizontal Flip Augmentation

`RandomFlip("horizontal")` is added to the preprocessing `Sequential`.


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Reshape, Conv2D, BatchNormalization, Activation,
    MaxPooling2D, Flatten, Dropout, Dense, RandomFlip,
)

from crest import Node, HierarchalTensorGraph
from crest.model.TensorSpec import TensorSpec

INTER = 'x'   # intermediate key between preprocessing and CNN nodes

# Node 1: preprocessing
# RandomFlip is a Keras augmentation layer - active during training only
preprocess_seq = Sequential([
    Reshape((patch_size, patch_size, n_channels)),
    RandomFlip("horizontal"),
])
preprocessing = Node(
    node=lambda X: {INTER: preprocess_seq(X[INP])},
    name='preprocessing',
    inputs={INP:    TensorSpec([None, patch_size, patch_size, n_channels])},
    outputs={INTER: TensorSpec([None, patch_size, patch_size, n_channels])},
)

# Node 2: CNN — Conv → BN → ReLU pattern with two spatial downsampling blocks
cnn_layers = Sequential([
    # First conv block: 32x32 -> 16x16
    Conv2D(32, 3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(64, 3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2),
    Dropout(0.25),
    # Second conv block: 16x16 -> 8x8
    Conv2D(128, 3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2),
    Dropout(0.25),
    # Dense head
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(n_classes, activation='softmax'),
])
cnn_node = Node(
    node=lambda X: {OUT: cnn_layers(X[INTER])},
    name='cnn',
    inputs={INTER: TensorSpec([None, patch_size, patch_size, n_channels])},
    outputs={OUT:  TensorSpec([None, n_classes])},
)

In [ ]:
graph = HierarchalTensorGraph(
    name='cifar10',
    inputs={INP: TensorSpec([None, patch_size, patch_size, n_channels])},
    outputs={OUT: TensorSpec([None, n_classes])},
)
graph.add_edge('input',       preprocessing)
graph.add_edge(preprocessing, cnn_node)
graph.add_edge(cnn_node,      'output')

graph.draw()

## 3. Compile and Train

`Model` wraps the HTG in a Keras functional model and exposes
`compile` / `fit` / `evaluate` / `predict` methods.

`steps_per_epoch=225` covers the full 45 000-sample training set each epoch
(45 000 ÷ 200 batch size). CIFAR-10 needs substantially more training than MNIST:
with Metal GPU this runs in a few minutes, reaching ~70 % test accuracy after 20 epochs.

In [ ]:
from crest.model.Model import Model
from crest.data.batching.Batcher import Batcher
from crest.utils.Metrics import Metrics

model = Model(graph)

# Register a custom CREST metric before compiling
def squared_diff(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

model.metrics.register('squared_diff', squared_diff)
print("Custom CREST metrics:", model.metrics.customs)

In [ ]:
# Compile with loss only — omit metrics= to avoid a TF/Keras crash
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
)

In [ ]:
batch_size = 200

batch_train = Batcher(ds_train, **{
    'batch_size': batch_size,
    'features'  : [[INP], [OUT]],  # inputs / targets
    'repeat'    : True,
    'workers'   : 0,
    'prefetch'  : False,
})

In [ ]:
# model.fit opens its own context manager for batch_train
history = model.fit(batch_train, **{
    'steps_per_epoch': 225,
    'epochs'         : 3,   # Increase to 20 for better accuracy but expect ~15 minutes of training time
    'verbose'        : 1,
})
batch_train.close()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history.history['loss']) + 1), history.history['loss'],
        color='steelblue', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training loss', fontsize=12)
ax.set_title('Training Loss over Epochs', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Evaluate

`model.evaluate` accepts a `Batcher` directly.
It wraps it as a plain Python iterator via `iter(batcher)` and forwards it to
`keras.Model.evaluate`.

Because `metrics=` is omitted from `compile`, `evaluate` returns only the
loss. Accuracy is computed manually from the raw model predictions: call
`model.predict` on the test set, take `np.argmax` over the class axis, and compare
with the ground-truth integer labels. The predicted class names are looked up in
`CIFAR10_CLASSES` for a human-readable summary.

In [ ]:
n_test       = len(ds_test.data[0])      # 10 000 test samples
n_steps_eval = n_test // batch_size      # 10 000 // 200 = 50

batch_test = Batcher(ds_test, **{
    'batch_size': batch_size,
    'features'  : [[INP], [OUT]],
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})
# model.evaluate manages the Batcher context internally
results = model.evaluate(batch_test, steps=n_steps_eval, verbose=0, return_dict=True)
print("Test loss:", results['loss'])

In [ ]:
# Compute accuracy manually
batch_test2 = Batcher(ds_test, **{
    'batch_size': batch_size,
    'features'  : [INP],
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})
preds_test = model.predict(batch_test2)
batch_test2.close()

y_pred_classes = np.argmax(preds_test[OUT], axis=-1)
y_true_classes = ds_test.data[1]   # integer labels
test_accuracy  = np.mean(y_pred_classes == y_true_classes)
print(f"Test accuracy : {test_accuracy:.4f}  "
      f"({int(test_accuracy * n_test)}/{n_test} correct)")

# Per-class accuracy
for i, cls in enumerate(CIFAR10_CLASSES):
    mask = y_true_classes == i
    acc  = np.mean(y_pred_classes[mask] == i)
    print(f"  {cls:<12}: {acc:.3f}")

### Evaluation Visualizations

Three complementary plots to understand where the model succeeds and where it fails:

- **Confusion matrix** — raw prediction counts; dominant off-diagonal entries reveal
  systematic class confusions (e.g. cat ↔ dog, bird ↔ airplane)
- **Per-class accuracy bar chart** — visual summary of the per-class accuracy table
  above
- **Sample predictions** — qualitative check; real test images annotated with true
  (T) and predicted (P) labels to sanity-check the failure modes

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm  = confusion_matrix(y_true_classes, y_pred_classes)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CIFAR10_CLASSES,
    yticklabels=CIFAR10_CLASSES,
    ax=ax,
)
ax.set_xlabel('Predicted class', fontsize=12)
ax.set_ylabel('True class', fontsize=12)
ax.set_title('Confusion Matrix — CIFAR-10 Test Set (10 000 samples)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
class_accs = [np.mean(y_pred_classes[y_true_classes == i] == i) for i in range(n_classes)]
colors     = ['#2ecc71' if a >= 0.75 else '#f39c12' if a >= 0.50 else '#e74c3c'
              for a in class_accs]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(CIFAR10_CLASSES, class_accs, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(y=test_accuracy, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Overall: {test_accuracy:.1%}')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Per-class Test Accuracy', fontsize=14)
ax.legend(fontsize=11)
for bar, acc in zip(bars, class_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 15 wrong + 10 correct predictions sampled from the test set
np.random.seed(42)
wrong_idx = np.where(y_pred_classes != y_true_classes)[0]
right_idx  = np.where(y_pred_classes == y_true_classes)[0]

sample_idx = np.concatenate([
    np.random.choice(wrong_idx, 15, replace=False),
    np.random.choice(right_idx, 10, replace=False),
])
np.random.shuffle(sample_idx)

fig, axes = plt.subplots(5, 5, figsize=(11, 11))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(ds_test.data[0][idx])
    true_cls = CIFAR10_CLASSES[y_true_classes[idx]]
    pred_cls = CIFAR10_CLASSES[y_pred_classes[idx]]
    correct  = y_true_classes[idx] == y_pred_classes[idx]
    color    = '#2ecc71' if correct else '#e74c3c'
    ax.set_title(f'T: {true_cls}\nP: {pred_cls}', color=color, fontsize=8, fontweight='bold')
    ax.axis('off')

fig.suptitle('Sample Predictions  (green = correct, red = wrong)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Predictions

`Model.predict` supports three calling styles:

- **Batcher** — CREST manages the context and concatenates batches internally;
  with `repeat=False` the batcher exhausts naturally (no `steps` needed here because
  `predict` uses its own loop rather than Keras's `EpochIterator`).
- **Raw dict** — pass `{input_name: numpy_array}` directly; useful for small ad-hoc
  inference without constructing a Batcher.
- Batcher with `coords` — extra keys (e.g. spatial coordinates or timestamps) are
  carried through the prediction step and returned alongside the model output.
  `coords` requires a `Batcher`; passing a plain dataset raises `ValueError`.

In [ ]:
# Predict with a Batcher
# repeat=False exhausts the dataset in one pass; no steps kwarg required.
batch_pred = Batcher(ds_pred, **{
    'batch_size': batch_size,
    'features'  : [INP],     # prediction-only set; no target column
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})

preds = model.predict(batch_pred)
batch_pred.close()

predicted_classes = np.argmax(preds[OUT], axis=-1)
predicted_names   = [CIFAR10_CLASSES[c] for c in predicted_classes]
print(f"Predictions shape  : {preds[OUT].shape}")
print(f"First 10 predicted : {predicted_classes[:10]}")
print(f"First 10 class names: {predicted_names[:10]}")
print(f"Class distribution : {np.bincount(predicted_classes)}")

In [ ]:
# Predict with a raw dict
# model.predict delegates to keras.Model.predict for non-Batcher inputs.
x_sample = ds_test.data[0][:10]   # first 10 test images, shape (10, 32, 32, 3)

preds_dict = model.predict({INP: x_sample})

print(f"Output keys : {list(preds_dict.keys())}")
print(f"Predicted   : {np.argmax(preds_dict[OUT], axis=-1)}")
print(f"Pred names  : {[CIFAR10_CLASSES[c] for c in np.argmax(preds_dict[OUT], axis=-1)]}")
print(f"True labels : {ds_test.data[1][:10]}")
print(f"True names  : {[CIFAR10_CLASSES[c] for c in ds_test.data[1][:10]]}")

In [ ]:
# Predict with extra coordinates
# Attach synthetic lat/lon metadata to each prediction sample.
# coords must be passed as a Batcher; using a plain dataset raises ValueError.
from crest.data.loading import StructuredDataset

x   = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)

ds_aux    = StructuredDataset(lat, lon, x, labels=['lat', 'lon', INP])
batch_aux = Batcher(ds_aux, **{
    'batch_size': batch_size,
    'features'  : ['lat', 'lon', INP],
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})

preds_with_coords = model.predict(batch_aux, coords=['lat', 'lon'])
batch_aux.close()

print("Keys in output:", list(preds_with_coords.keys()))
print("lat (first 5): ", preds_with_coords['lat'][:5])
print("lon (first 5): ", preds_with_coords['lon'][:5])

## 6. Save and Reload

`Model.save(filepath)` writes a zip archive containing:

- **graph.json** — the HTG encoded with `dill` and base64, plus the registry name used
  to reconstruct the class on load
- **model.weights.h5** — Keras trainable weights
- **metrics.json** — the CREST `Metrics` registry (including custom metrics) via
  `Metrics.to_json` / `jsonpickle`
- **compile_args.json** — Keras compile configuration

`Model.load(filepath)` is a `classmethod` that reverses the process: it looks up the
HTG class by name in HTG registry, decodes the graph, restores weights, and re-compiles.

In [ ]:
import os

model.save('cifar10.crest')
loaded_model = Model.load('cifar10.crest')

# Verify loaded model gives identical predictions
pred_orig   = model.predict({INP: x_sample})[OUT]
pred_loaded = loaded_model.predict({INP: x_sample})[OUT]
max_diff    = np.abs(pred_orig - pred_loaded).max()

print(f"Max prediction difference after reload: {max_diff:.2e}")
print("Custom metrics preserved:", loaded_model.metrics.customs)

os.remove('cifar10.crest')